In [1]:
library(tidyverse)
library(dplyr)

── Attaching core tidyverse packages ───── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ─────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
nhl_model_ready <- readRDS("nhl_model_ready.rds")
nhl_outcomes <- readRDS ("nhl_outcomes.rds")

nrow(nhl_model_ready)
nrow(nhl_outcomes)

[1] 9972

[1] 5206

In [4]:
training_data <- nhl_outcomes |>
    filter(season_start <= 2022)

validation_data <- nhl_outcomes |>
    filter(season_start == 2023)

testing_data <- nhl_outcomes |>
    filter(season_start == 2024)

prediction_data <- nhl_model_ready |>
    filter(season_start == 2025)

In [6]:
range(training_data$season_start)
unique(validation_data$season_start)
unique(testing_data$season_start)
unique(prediction_data$season_start)

[1] 2015 2022

[1] 2023

[1] 2024

[1] 2025

In [7]:
table(training_data$breakout)


   0    1 
3678  434 

In [10]:
training_model <- training_data |>
    mutate(position = as.factor(position)) |>
    drop_na(
        breakout, 
        points_per_82,
        average_minutes,
        games_played,
        age,
        position
        )

In [11]:
breakout_model <- glm(
    breakout ~
        points_per_82 +
        average_minutes +
        games_played +
        age +
        position,
    data = training_model,
    family = binomial
    )

#breakout probability is predicted using previous scoring, ice time,
#games, age, and position 

In [13]:
summary(breakout_model)


Call:
glm(formula = breakout ~ points_per_82 + average_minutes + games_played + 
    age + position, family = binomial, data = training_model)

Coefficients:
                 Estimate Std. Error z value Pr(>|z|)    
(Intercept)     -0.704431   0.484887  -1.453  0.14629    
points_per_82   -0.035100   0.004591  -7.645 2.09e-14 ***
average_minutes  0.251893   0.031596   7.972 1.56e-15 ***
games_played    -0.012416   0.003864  -3.213  0.00131 ** 
age             -0.117854   0.014511  -8.122 4.59e-16 ***
positionD       -2.093797   0.244202  -8.574  < 2e-16 ***
positionL        0.322621   0.140746   2.292  0.02189 *  
positionR        0.444282   0.148957   2.983  0.00286 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 2772.3  on 4111  degrees of freedom
Residual deviance: 2600.8  on 4104  degrees of freedom
AIC: 2616.8

Number of Fisher Scoring iterations: 5


In [14]:
exp(coef(breakout_model))

(Intercept)   points_per_82 average_minutes    games_played             age 
      0.4943897       0.9655087       1.2864582       0.9876610       0.8888257 
      positionD       positionL       positionR 
      0.1232184       1.3807424       1.5593700

In [16]:
validation_model <- validation_data |>
    mutate (
        position = factor(
            position,
            levels = levels(training_model$position)
            )) |>
    drop_na(
        breakout,
        points_per_82,
        average_minutes,
        games_played,
        age,
        position
        )

In [17]:
validation_predictions <- validation_model |>
    mutate(
        breakout_probability = predict (
            breakout_model,
            newdata = validation_model,
            type = "response"))

In [19]:
validation_predictions |>
    arrange(desc(breakout_probability))|>
    select(
        player_name,
        age,
        position,
        points_per_82,
        average_minutes,
        breakout_probability,
        breakout
        ) |>
    slice_head(n = 20) |>
    print (n = 20)

# A tibble: 20 × 7
   player_name   age position points_per_82 average_minutes breakout_probability
   <chr>       <dbl> <fct>            <dbl>           <dbl>                <dbl>
 1 Barrett Ha…  23.3 C                 24.8            17.0                0.390
 2 Leo Carlss…  18.8 C                 43.2            18.1                0.365
 3 Tyson Foer…  21.7 R                 35.1            17.3                0.341
 4 Matt Coron…  20.9 R                 21.7            12.6                0.326
 5 Taylor Rad…  25.6 R                 15.7            15.9                0.324
 6 William Ek…  21.0 L                 46.1            18.7                0.318
 7 Trevor Zeg…  22.5 C                 39.7            17.1                0.303
 8 Lukas Reic…  21.4 L                 20.2            14.1                0.297
 9 Juraj Slaf…  19.5 L                 50              17.9                0.282
10 Matty Beni…  20.9 C                 39.4            18.0                0.272
11 Zach B

In [20]:
validation_predictions |>
    filter(breakout == 1) |>
    arrange(desc(breakout_probability))|>
    select (
        player_name,
        team,
        points_per_82,
        next_points_per_82,
        points_increase,
        breakout_probability
        ) |>
    print(n = Inf)

# A tibble: 41 × 6
   player_name        team  points_per_82 next_points_per_82 points_increase
   <chr>              <chr>         <dbl>              <dbl>           <dbl>
 1 Barrett Hayton     ARI            24.8               46              21.2
 2 Matt Coronato      CGY            21.7               50.1            28.3
 3 William Eklund     SJS            46.1               61.8            15.6
 4 Kaapo Kakko        NYR            25.5               45.7            20.1
 5 Dylan Holloway     EDM            19.4               67.1            47.7
 6 Alex Laferriere    LAK            23.3               44.7            21.4
 7 Kirill Marchenko   CBJ            44.2               76.8            32.7
 8 Kent Johnson       CBJ            31.2               68.7            37.5
 9 Simon Holmstrom    NYI            27.3               49.2            21.9
10 Tom Wilson         WSH            38.8               65.8            27.0
11 JJ Peterka         BUF            50                 7